# ML basics on old data

**Goal:** train your first model, understand every piece of it, and notice something suspicious.

Read `docs/ml_basics.md` alongside this notebook.

## 1. A toy problem first: stars vs asteroids from one number
Stars do not move. Asteroids move. Suppose we measure each object's speed in arcsec/hour,
and the measurement is noisy. Can we *learn* the cut instead of guessing it?

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(42)

# 200 stars: true speed 0, measurement noise of about 1 "/hr
star_speed = np.abs(rng.normal(0, 1.0, 200))
# 200 asteroids: true speeds between 2 and 40 "/hr, plus noise
asteroid_speed = rng.uniform(2, 40, 200) + rng.normal(0, 1.0, 200)

speed = np.concatenate([star_speed, asteroid_speed]).reshape(-1, 1)   # features: shape (400, 1)
label = np.array([0] * 200 + [1] * 200)                              # labels: 0 = star, 1 = asteroid

plt.hist(star_speed, bins=40, alpha=0.6, label="stars (0)")
plt.hist(asteroid_speed, bins=40, alpha=0.6, label="asteroids (1)")
plt.xlabel('measured speed ("/hr)'); plt.legend(); plt.show()

In [ ]:
from asthunt.models.train_baseline_classifier import train_baseline_classifier

toy_model = train_baseline_classifier(speed, label)

# The model outputs a probability, not just a yes/no
test_speeds = np.array([[0.5], [1.5], [2.5], [5.0], [20.0]])
for s, p in zip(test_speeds[:, 0], toy_model.predict_proba(test_speeds)[:, 1]):
    print(f"speed {s:5.1f} \"/hr  ->  p(asteroid) = {p:.3f}")

The model learned a rule by itself. The curve it learned is the **sigmoid**:
`p = 1 / (1 + exp(-(w * x + b)))`. Training chose `w` and `b`.

Notice that the 50% point sits near **5 "/hr**, even though the classes separate around 2 "/hr. Two reasons:
1. **Regularization.** By default scikit-learn penalises big weights (setting `C=1.0`). Big `w` = steep curve, so the penalty keeps the curve gentle.
   That protects against overfitting noisy data, but here it blurs a clean boundary.
2. **One straight line for very spread-out data.** Asteroid speeds go up to 40, which pulls the fit.

Try exercise 4 at the bottom to see the curve sharpen.

In [ ]:
x_grid = np.linspace(0, 10, 200).reshape(-1, 1)
plt.plot(x_grid, toy_model.predict_proba(x_grid)[:, 1])
plt.axhline(0.5, ls="--", c="gray")
plt.xlabel('speed ("/hr)'); plt.ylabel("p(asteroid)"); plt.title("the learned sigmoid"); plt.show()

scaler, logreg = toy_model[0], toy_model[1]
print("learned weight w =", logreg.coef_[0][0].round(2), "  bias b =", logreg.intercept_[0].round(2))
print("(these are in *scaled* units, because the scaler ran first)")

## 2. Now the real project data: `legacy/tracklets.csv`
Each row is one tracklet: 4 detections linked across 4 frames. `label` is 1 for an injected (real) asteroid, 0 for bogus.

In [ ]:
import pandas as pd
from asthunt.config.load_config import PROJECT_ROOT

tracklets = pd.read_csv(PROJECT_ROOT / "legacy" / "tracklets.csv")
print(tracklets.shape)
tracklets.head()

### Which columns are allowed to be features?
A feature must be something we could **measure on new, unlabelled data**.
- `mag`, `true_rate`, `true_angle` are the *injection settings*. We only know them because we made the fakes. Never features.
- `tracklet_id` is just a row number. Never a feature.
- `measured_rate`, `linearity_residual` are measured from the detections. Allowed.

Look what happens with `mag`:

In [ ]:
print(tracklets.groupby("label")["mag"].apply(lambda m: m.isna().mean()).rename("fraction of mag that is missing"))

`mag` is missing for **every** bogus row and present for **every** real one. A model given `mag` would learn
"missing → bogus" and be 100% right for a useless reason. This is **target leakage**: the label sneaks in through a feature.

## 3. Train / test split
The model learns from the **train** part. We judge it only on the **test** part it has never seen.

In [ ]:
from sklearn.model_selection import train_test_split

feature_columns = ["measured_rate", "linearity_residual"]
X = tracklets[feature_columns].to_numpy()
y = tracklets["label"].to_numpy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y   # stratify keeps the 50/50 ratio in both parts
)
print("train rows:", len(X_train), "  test rows:", len(X_test))

In [ ]:
model = train_baseline_classifier(X_train, y_train)

print("train accuracy:", model.score(X_train, y_train))
print("test accuracy :", model.score(X_test, y_test))

In [ ]:
from sklearn.metrics import confusion_matrix

print("rows = truth (bogus, real), columns = prediction (bogus, real)")
print(confusion_matrix(y_test, model.predict(X_test)))

## 4. A near-perfect score. Should we celebrate?
99.8% on unseen data. **No celebration yet.** Plot the data and see *why* it is so high:

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for value, name in [(0, "bogus"), (1, "real")]:
    rows = tracklets[tracklets["label"] == value]
    axes[0].hist(np.log10(rows["linearity_residual"]), bins=50, alpha=0.6, label=name)
    axes[1].hist(np.log10(rows["measured_rate"]), bins=50, alpha=0.6, label=name)
axes[0].set_xlabel("log10(linearity residual, px)"); axes[0].legend()
axes[1].set_xlabel('log10(measured rate, "/hr)'); axes[1].legend()
plt.show()

The two classes don't even touch. Smallest bogus residual vs largest real residual:

In [ ]:
print("largest  real  residual:", round(tracklets.loc[y == 1, "linearity_residual"].max(), 2), "px")
print("smallest bogus residual:", round(tracklets.loc[y == 0, "linearity_residual"].min(), 2), "px")

A one-line rule (`residual < 100 px`) gets **100%**, better than our model! Two lessons in one:

1. **The model's mistakes come from the model, not from hard data.** Logistic regression draws a straight line in the *raw* numbers,
   and these numbers span 0.004 to 2300. On a log scale the separation is obvious (exercise 1).
   Choosing how to present features to a model is called **feature engineering**.
2. **The data is far too easy.** Bogus examples were built from 4 **random** sources anywhere on the image, so they are hundreds
   of pixels off a straight line. Real bogus tracklets are *near misses*, a few pixels off. Our test set came from the same easy
   data, so even 100% tells us nothing about the real sky.

**Lesson:** a (near-)perfect score is a reason to check your data, not a reason to celebrate. The next phase (*Leakage audit*) digs into this.

## Exercises
1. Train on `np.log10(X)` instead of `X`. What's the test accuracy now? Why does a log transform help a *linear* model?
2. Train using only `linearity_residual`. Then only `measured_rate`. Which one carries the signal?
3. Add `mag` as a feature (fill missing with 0). What happens, and why is it cheating?
4. In the toy problem, build `LogisticRegression(C=1000)` yourself (with the scaler). Where is the 50% point now?